# Train whole-resistor segmentation in Google Colab

Use a GPU runtime and choose **Runtime → Run all**. This trains a binary resistor/background model with torchvision **LR-ASPP + MobileNetV3-Large**. It does not use Ultralytics.

The notebook now bootstraps through a fresh `git clone` of `main` instead of `raw.githubusercontent.com`, avoiding stale CDN-cached trainer scripts. At startup it prints an explicit `Segmentation Colab version:` line so you can verify the current code is running.

The dataset is downloaded automatically from `m2/detection_res.v1i.coco-segmentation.zip`. Both source classes `resistor` and `res` are collapsed into one foreground class. All persistent outputs go to `MyDrive/resistor_model/segmentation-m2-lraspp`.

Training prints every batch with epoch, batch %, loss, learning rate and elapsed time. Validation/test output includes Dice and IoU. During any silent startup work the notebook prints a trainer heartbeat every 5 seconds, so the cell never appears frozen.

If `last.pt` exists in Drive, it is copied first to local `/content` with visible copy progress and training resumes from that local copy. `last.pt` is saved after every epoch, `best.pt` whenever validation Dice improves, and `checkpoints/epoch_NNN.pt` every 5 epochs plus the final epoch. Optimizer, scheduler and AMP scaler state are included.

Default training: 384×384, 60 epochs, batch size 16, weighted cross-entropy + foreground Dice loss. The MobileNetV3 backbone trains from scratch by default, avoiding torchvision's ImageNet-pretrained weights and their potentially dataset-derived terms. Set `USE_PRETRAINED_BACKBONE = True` in the downloaded script only after reviewing those terms for your use case.

After training, the best checkpoint is exported to ONNX and the complete persistent run directory is packaged as `resistor-segmentation-m2-lraspp.zip` for download. If browser download fails, everything remains in Drive.

The exported ONNX returns a single `[N,1,H,W]` resistor probability map for mobile post-processing: threshold → connected component → long-axis/oriented crop → ResistorBandNet.

In [ ]:
import pathlib, shutil, subprocess
bootstrap = pathlib.Path('/content/resistor_model_bootstrap')
if bootstrap.exists():
    shutil.rmtree(bootstrap)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/resistor_model.git', str(bootstrap)], check=True)
subprocess.run(['git', '-C', str(bootstrap), 'rev-parse', '--short', 'HEAD'], check=True)
script = bootstrap / 'colab' / 'train_resistor_segmentation_colab.py'
source = script.read_text(encoding='utf-8')
if 'SEGMENTATION_COLAB_VERSION' not in source or 'PROGRESS_EVERY = 1' not in source or 'run_with_heartbeat' not in source:
    raise RuntimeError('Fresh clone does not contain the expected current Colab trainer. Stop instead of running stale code.')
exec(compile(source, str(script), 'exec'), {'__name__': '__main__'})
